从 ocr 结果，结合职官表，制作表格（主要区分关键字与非关键字）

In [19]:
import json
with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "r", encoding="utf-8") as f:
  bureaucracy_items = json.load(f)

"""制作职官关键字表 称呼 + 页码"""
bureaucracy_keywords = set()
bklist = []

fs = False
for item in bureaucracy_items:
  if not fs:
    if item["text"] == "第八编 军事统率机构与地方治安机构类":
      fs = True
    else:
      continue
  if item["text"] == "第十一编 阶官类":
    break
  if item["type"] == "name":
    bureaucracy_keywords.add(item["text"])
    bklist.append(item)
print(len(bureaucracy_keywords))


806


In [2]:
attribute_dict = {
  "简称与别名": ["简称"],
  "简称与别称": ["简称"],
  "职源、沿革、职掌、品位": ["职源", "职掌", "官品"],
  "职源与沿革、职掌、官品": ["职源", "职掌", "官品"], 
  "职源与沿革、职掌": ["职源", "职掌"],
  "官品、编制、简称与别名": ["官品", "编制", "简称"],
  "职源、沿革、编制": ["职源", "编制"],
  "职掌与沿革": ["职掌", "职源"],
  "职源、职掌、编制": ["职源", "职掌", "编制"], 
  "职掌、官品、编制": ["职掌", "官品", "编制"], 
  "职源、职掌": ["职源", "职掌"],
  "职掌、品位": ["职掌", "官品"], 
  "编制、职能":["职掌", "编制"],
  "编制与品位": ["编制", "官品"],
  "沿革与职掌": ["职源", "职掌"],
  "省称与别名": ["简称"],
  "简称与追改": ["简称"],
  "简称与旧称": ["简称"],
  "职源与沿革": ["职源"], 
  "职源与改革": ["职源"],
  "省称与别名": ["简称"],
  "追称": ["简称"],
  "职掌": ["职掌"],
  "职能": ["职掌"],
  "位遇": ["官品"],
  "序位": ["官品"], 
  "地位": ["官品"],
  "品秩": ["官品"],
  "编制": ["编制"],
  "职源": ["职源"], 
  "简称": ["简称"], 
  "通称": ["简称"], 
  "省称": ["简称"], 
  "别名": ["简称"], 
  "别称": ["简称"], 
  "合称": ["简称"],
  "品阶": ["官品"],
  "官品": ["官品"], 
  "品位": ["官品"],
  "班位": ["官品"],
  "沿革": ["职源"],
  "泛称": ["简称"]
}

attribute_keywords = set()
for item in attribute_dict.keys():
  attribute_keywords.add(item)

print(len(attribute_keywords))

41


In [3]:
""" Trie Tree 实现多关键字查找 """
class TrieNode:
    def __init__(self):
        self.children = {}  # 子节点字典，key是字符，value是TrieNode
        self.is_end = False  # 标记是否是某个关键字的结尾
        self.keyword = None  # 存储完整的关键字（用于返回）

class TrieTree:
    def __init__(self):
        self.root = TrieNode()
    
    def insert(self, keyword):
        """插入关键字到 Trie 树"""
        node = self.root
        for char in keyword:
            if char not in node.children:
                node.children[char] = TrieNode()
            node = node.children[char]
        node.is_end = True
        node.keyword = keyword
    
    def start_with(self, text):
        """
        判断 text 是否以某个关键字开头
        返回匹配的关键字，如果没有匹配则返回 None
        如果有多个匹配，返回最长的那个（因为关键字按长度从长到短插入）
        """
        if not text:
            return None
            
        node = self.root
        matched_keyword = None
        
        for i, char in enumerate(text):
            if char not in node.children:
                # 无法继续匹配，返回之前找到的最长匹配
                break
            node = node.children[char]
            if node.is_end:
                # 找到匹配的关键字，记录（因为关键字按长度排序，后面可能还有更长的）
                matched_keyword = node.keyword
        
        return matched_keyword
    
    def build_from_set(self, keyword_set):
        """从关键字集合构建 Trie 树"""
        # 按长度从长到短排序，确保优先匹配长关键字
        sorted_keywords = sorted(keyword_set, key=len, reverse=True)
        for keyword in sorted_keywords:
            self.insert(keyword)



In [5]:
# 构建 bureaucracy 的 Trie 树
trie_tree_bureaucracy = TrieTree()
trie_tree_bureaucracy.build_from_set(bureaucracy_keywords)
print(f"Bureaucracy Trie 树构建完成，包含 {len(bureaucracy_keywords)} 个关键字")

# 构建 attribute 的 Trie 树
trie_tree_attribute = TrieTree()
trie_tree_attribute.build_from_set(attribute_keywords)
print(f"Attribute Trie 树构建完成，包含 {len(attribute_keywords)} 个关键字")

# 定义查找函数
def start_with_bureaucracy(text):
    """判断 text 是否以某个 bureaucracy 关键字开头"""
    return trie_tree_bureaucracy.start_with(text)

def start_with_attribute(text):
    """判断 text 是否以某个 attribute 关键字开头"""
    return trie_tree_attribute.start_with(text)

# 测试
test_text1 = """提点纲马驿程
差遣名。隶都大提举茶马
司。掌茶马司起发纲马驿程公事，即负责长途运
送纲马事。南宋时，自成都府至兴元府驿程置提
点官一员，自兴元府至汉阳军一员，自汉阳军至临
安一员。又，广西路置提点纲马官二员，自静江府
至抚州一员，自抚州至临安一员。由武臣大使臣
以上、通晓马政人充（《宋会要·职官》43之117、
118,112)。"""
test_text2 = "职掌 掌筹措军马钱粮。"
print(f"\n测试1: '{test_text1}'")
print(f"  Bureaucracy匹配: {start_with_bureaucracy(test_text1)}")
print(f"  Attribute匹配: {start_with_attribute(test_text1)}")
print(f"\n测试2: '{test_text2}'")
print(f"  Bureaucracy匹配: {start_with_bureaucracy(test_text2)}")
print(f"  Attribute匹配: {start_with_attribute(test_text2)}")


Bureaucracy Trie 树构建完成，包含 4818 个关键字
Attribute Trie 树构建完成，包含 41 个关键字

测试1: '提点纲马驿程
差遣名。隶都大提举茶马
司。掌茶马司起发纲马驿程公事，即负责长途运
送纲马事。南宋时，自成都府至兴元府驿程置提
点官一员，自兴元府至汉阳军一员，自汉阳军至临
安一员。又，广西路置提点纲马官二员，自静江府
至抚州一员，自抚州至临安一员。由武臣大使臣
以上、通晓马政人充（《宋会要·职官》43之117、
118,112)。'
  Bureaucracy匹配: 提点纲马驿程
  Attribute匹配: None

测试2: '职掌 掌筹措军马钱粮。'
  Bureaucracy匹配: None
  Attribute匹配: 职掌


In [ ]:
"""标题问题"""
import json
with open("../../data/ocr-results/第八至十编OCR提取结果文本.json", "r", encoding="utf-8") as f:
  page_data = json.load(f)

with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "r", encoding="utf-8") as f:
  records = json.load(f)
titles = []
for record in records:
  if record["type"] in ["h1", "h2", "h3", "catalog"]:
    titles.append(record["text"])

for page in page_data:
  for content in page["contents"]:
    if content["type"] == "title":
      # if content["text"] not in titles:
      print(content["text"])

第八编 军事统率机构与地方治安机构类
一、大元帅府、都督府门
二、兵马都部署、钤辖、监押与巡检门
三、制置、宣谕、招讨、经略安抚使门
四、宣抚司、总领所门
五、御前诸军都统制司门
六、将司门
第九编 地方官类之一——路官
一、总监司门
二、发运使、转运使门
三、提点刑狱公事门
四、提举常平公事门
五、监、冶、场、务门
六、市舶司门
七、安抚使、经总制司门
八、提举保甲司、学事司、察访司等路机构门
第十编 地方官类之二——府州县官
一、京府、次府、留守司门
二、州、府、军、监门
[附]邮置
三、幕职官与诸曹官门
四、州府学与书院门
五、县镇官与监当官门


In [120]:
"""转化正文部分"""
import json
import json
with open("../../data/ocr-results/职官条目分类目录-catalog-refined.json", "r", encoding="utf-8") as f:
  bureaucracy_items = json.load(f)

bureaucracy_keywords = set()
bklist = []
fs = False
for item in bureaucracy_items:
  if not fs:
    if item["text"] == "第八编 军事统率机构与地方治安机构类":
      fs = True
    else:
      continue
  if item["text"] == "第十一编 阶官类":
    break
  if item["type"] == "name":
    bureaucracy_keywords.add(item["text"])
    bklist.append(item)
print(len(bklist))

# 构建 bureaucracy 的 Trie 树
trie_tree_bureaucracy = TrieTree()
trie_tree_bureaucracy.build_from_set(bureaucracy_keywords)
print(f"Bureaucracy Trie 树构建完成，包含 {len(bureaucracy_keywords)} 个关键字")

# 定义查找函数
def start_with_bureaucracy(text):
    """判断 text 是否以某个 bureaucracy 关键字开头"""
    return trie_tree_bureaucracy.start_with(text)

def start_with_attribute(text):
    """判断 text 是否以某个 attribute 关键字开头"""
    return trie_tree_attribute.start_with(text)

with open("../../data/ocr-results/第八至十编OCR提取结果文本-refined.json", "r", encoding="utf-8") as f:
  page_data = json.load(f)

blocks = []
for page in page_data:
  for content in page["contents"]:
    if content["type"] == "text":
      b = start_with_bureaucracy(content["text"])
      a = start_with_attribute(content["text"])
      if b:
        blocks.append({
          "type": "bureaucracy_name",
          "text": b,
        })
        blocks.append({
          "type": "text",
          "text": content["text"][len(b):],
        })
      elif a:
        blocks.append({
          "type": "attribute",
          "text": a,
        })
        blocks.append({
          "type": "text",
          "text": content["text"][len(a):],
        })
      else:
        blocks.append({
          "type": "text",
          "text": content["text"],
        })

with open("../../data/ocr-results/第八至十编-结构化-20260110.json", "w", encoding="utf-8") as f:
  json.dump(blocks, f, ensure_ascii=False, indent=2)

eblist = []
for b in blocks:
  if b["type"] == "bureaucracy_name":
    eblist.append(b)

print(len(bklist), len(eblist))
print(bklist[:10])
print(eblist[:10])
print("===")

for i in range(len(bklist)):
  if bklist[i]["text"] != eblist[i]["text"]:
    print(i)
    print(json.dumps(bklist[i-1:i+1], ensure_ascii=False, indent=2))
    print(json.dumps(eblist[i], ensure_ascii=False, indent=2))
    break

832
Bureaucracy Trie 树构建完成，包含 823 个关键字
832 832
[{'type': 'name', 'text': '河北兵马大元帅府', 'page': '482'}, {'type': 'name', 'text': '河北兵马大元帅', 'page': '482'}, {'type': 'name', 'text': '河北兵马元帅', 'page': '482'}, {'type': 'name', 'text': '河北兵马副元帅', 'page': '482'}, {'type': 'name', 'text': '河北兵马大元帅府参议官', 'page': '482'}, {'type': 'name', 'text': '河北兵马大元帅府随军应副', 'page': '482'}, {'type': 'name', 'text': '河北兵马大元帅府都统制五军兵马', 'page': '482'}, {'type': 'name', 'text': '河北兵马大元帅府统制', 'page': '482'}, {'type': 'name', 'text': '都督府', 'page': '483'}, {'type': 'name', 'text': '江淮荆浙都督府', 'page': '483'}]
[{'type': 'bureaucracy_name', 'text': '河北兵马大元帅府'}, {'type': 'bureaucracy_name', 'text': '河北兵马大元帅'}, {'type': 'bureaucracy_name', 'text': '河北兵马元帅'}, {'type': 'bureaucracy_name', 'text': '河北兵马副元帅'}, {'type': 'bureaucracy_name', 'text': '河北兵马大元帅府参议官'}, {'type': 'bureaucracy_name', 'text': '河北兵马大元帅府随军应副'}, {'type': 'bureaucracy_name', 'text': '河北兵马大元帅府都统制五军兵马'}, {'type': 'bureaucracy_name', 'text': '河北兵马大元帅府统制'}, {'t

In [131]:
a = set([e["text"] for e in eblist])
print(len(a), len(eblist))

with open("../../data/ocr-results/第八至十编-结构化-20260110.json", "r", encoding="utf-8") as f:
  records = json.load(f)

bureaucrary_list = []
item = None
current_attribute = None
i = 0
while i < len(records):
  record = records[i]
  if record["type"] == "bureaucracy_name":
    item = {
      "name": record["text"],
      "texts": []
    }
    bureaucrary_list.append(item)
    current_attribute = "texts"
  elif record["type"] == "attribute":
    current_attribute = record["text"]
    item[current_attribute] = ""
  elif record["type"] == "text":
    if current_attribute == "texts":
      item["texts"].append(record["text"].strip())
    else:
      item[current_attribute] += record["text"].strip()
  i+=1
print(len(bureaucrary_list))

for item in bureaucrary_list:
  # if len(item["texts"]) > 1:
  #   print(item["texts"])
  item["text"] = "".join(item["texts"])
  del item["texts"]


with open("../../data/ocr-results/第八至十编-表格化结果.json", "w", encoding="utf-8") as f:
  json.dump(bureaucrary_list, f, ensure_ascii=False, indent=2)

823 832
832


In [132]:
attributes = set()
for record in records:
  if record["type"] == "attribute":
    attributes.add(record["text"])
print(json.dumps(list(attributes), ensure_ascii=False, indent=2))

[
  "追称",
  "省称",
  "职源",
  "职源与沿革",
  "品位",
  "职能",
  "官品",
  "别名",
  "别称",
  "编制与品位",
  "编制",
  "简称",
  "简称与别名",
  "职掌"
]


In [133]:
import json
import csv
from collections import OrderedDict

# 读取 JSON 文件
with open("../../data/ocr-results/第八至十编-表格化结果.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# 收集所有可能的列名
all_columns = set()
for item in data:
    for key in item.keys():
        all_columns.add(key)

# 定义列的顺序：条目名、说明文字优先，其他按字母顺序
columns = ["条目名", "说明文字"]
other_columns = sorted([col for col in all_columns if col not in ["name", "text"]])
columns.extend(other_columns)

# 转换为 CSV 格式
csv_rows = []
for item in data:
    row = OrderedDict()
    # name -> 条目名
    row["条目名"] = item.get("name", "")
    # text -> 说明文字
    row["说明文字"] = item.get("text", "")
    # 其他属性保持原样
    for col in other_columns:
        if col in item:
            row[col] = item[col]
        else:
            row[col] = ""
    csv_rows.append(row)

# 写入 CSV 文件
output_file = "../../data/ocr-results/第八至十编-表格化结果.csv"
with open(output_file, "w", encoding="utf-8-sig", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=columns)
    writer.writeheader()
    writer.writerows(csv_rows)

print(f"转换完成！共 {len(csv_rows)} 条记录")
print(f"CSV 文件已保存到: {output_file}")
print(f"列数: {len(columns)}")
print(f"列名: {', '.join(columns[:10])}..." if len(columns) > 10 else f"列名: {', '.join(columns)}")


转换完成！共 832 条记录
CSV 文件已保存到: ../../data/ocr-results/第八至十编-表格化结果.csv
列数: 16
列名: 条目名, 说明文字, 别名, 别称, 品位, 官品, 省称, 简称, 简称与别名, 编制...
